## Radar Student V3 — multi-stream

Three independent streams, fused only after independent encoding:
**point cloud** (PointNet, max+mean pooling) · **Range-Doppler map** ·
**Range-Angle map** — each map expanded to 5 channels (original, two spatial
gradients, gradient magnitude, temporal gradient).

Streams activate by what the export actually carries: v1 = points only;
radar_people_ra.cfg sessions add RA; RD needs dedicated slow sessions (it is
off in the live cfg — 32 KB/frame does not fit the link). A missing map
disables its stream instead of stopping the run. Ablations cover point
features always, and map channels whenever the maps exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/thermal-fusion/gexport/v2'

In [ ]:
import json, os, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

with open(os.path.join(DATA, 'manifest.json')) as f:
    MAN = json.load(f)
print(json.dumps(MAN['split'], indent=1))
TH_W, TH_H = MAN['thermal']['w'], MAN['thermal']['h']
RGB_W, RGB_H = 640, 400
RADAR_K = MAN['radar']['k']
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def validate_source_provenance():
    fields = ('lepton_gain', 'warp_lut_sha256',
              'radar_calib_sha256', 'radar_cfg_sha256',
              'detector_model', 'detector_engine_sha256')
    for field in fields:
        values = {info.get('provenance', {}).get(field)
                  for info in MAN['sessions'].values()} - {None}
        if len(values) > 1:
            raise ValueError(f'incompatible session provenance for {field}: {values}')
    unknown_cfg = [s for s, info in MAN['sessions'].items()
                   if not info.get('provenance', {}).get('radar_cfg_sha256')]
    if unknown_cfg:
        print(f'[WARN] radar config provenance unknown for {len(unknown_cfg)} session(s)')

validate_source_provenance()

def validate_thermal_scales(which):
    modes = {'celsius' if MAN['sessions'][s].get('c_per_lsb') else 'unit'
             for s in MAN['split'][which]}
    if len(modes) > 1:
        raise ValueError(f'{which} mixes calibrated Celsius and uncalibrated unit thermal sessions: {modes}')
    return next(iter(modes), 'empty')

def to_celsius_or_unit(thermal_raw, sess):
    """Raw counts -> degrees C when calibrated, otherwise session-relative [0,1].
    Mixing calibrated and uncalibrated sessions in one split is rejected."""
    info = MAN['sessions'][sess]
    if info.get('c_per_lsb'):
        return thermal_raw.astype(np.float32) * info['c_per_lsb'] + info['tmin']
    return thermal_raw.astype(np.float32) / float(info['thermal_counts_max'])

def load_split(which):
    validate_thermal_scales(which)
    out = {'thermal': [], 'dt_ms': [], 'radar': [], 'n_radar': [],
           'th_boxes': [], 'n_th_boxes': [], 'rgb_boxes': [], 'n_rgb_boxes': [],
           'scaled': [], 'range_angle': [], 'range_doppler': [],
           'ra_valid': [], 'rd_valid': [],
           'thermal_label_state': [], 'radar_label_state': []}
    for sess in MAN['split'][which]:
        for p in sorted(glob.glob(os.path.join(DATA, f'{sess}-*.npz'))):
            z = np.load(p)
            for k in out:
                if k == 'scaled':
                    out[k].append(to_celsius_or_unit(z['thermal'], sess))
                elif k in z:
                    out[k].append(z[k])
    # a key missing from ANY shard is dropped whole: a split where
    # only some sessions carry maps must not fake the rest as zeros
    n_shards = len(out['thermal'])
    return {k: np.concatenate(v) for k, v in out.items()
            if v and len(v) == n_shards}

## Thermal Student

Channels: 0 intensity/temperature · 1–2 grad-x/y · 3 |∇| · 4 Δ-from-median ·
5 temporal diff (zeroed when dt_ms>300 — an FFC gap must not read as motion).

In [ ]:
N_CH = 6
CH_NAMES = ['intensity', 'grad_x', 'grad_y', 'grad_mag', 'delta_bg', 'temporal']
STRIDE = 4
HM_W, HM_H = TH_W // STRIDE, TH_H // STRIDE

def build_channels(t, dt_ms):
    gx = np.gradient(t, axis=2); gy = np.gradient(t, axis=1)
    gmag = np.sqrt(gx**2 + gy**2)
    dbg = t - np.median(t, axis=(1, 2), keepdims=True)
    tdiff = np.zeros_like(t)
    ok = (dt_ms[1:] > 0) & (dt_ms[1:] < 300.0)
    tdiff[1:][ok] = t[1:][ok] - t[:-1][ok]
    return np.stack([t, gx, gy, gmag, dbg, tdiff], axis=1)

def make_targets(th_boxes, n_boxes):
    N = len(n_boxes)
    hm = np.zeros((N, 1, HM_H, HM_W), np.float32)
    wh = np.zeros((N, 2, HM_H, HM_W), np.float32)
    mask = np.zeros((N, 1, HM_H, HM_W), np.float32)
    yy, xx = np.mgrid[0:HM_H, 0:HM_W]
    for i in range(N):
        for b in range(int(n_boxes[i])):
            x, y, w, h = th_boxes[i, b, :4]
            cx, cy = int((x + w/2)/STRIDE), int((y + h/2)/STRIDE)
            if 0 <= cx < HM_W and 0 <= cy < HM_H:
                s = max(1.0, (w + h)/(4*STRIDE))
                hm[i, 0] = np.maximum(hm[i, 0], np.exp(-((xx-cx)**2 + (yy-cy)**2)/(2*s*s)))
                wh[i, :, cy, cx] = [w/TH_W, h/TH_H]
                mask[i, 0, cy, cx] = 1
    return hm, wh, mask

def augment_thermal(X, hm, wh, m, rng):
    """Per-batch: mirror (grad_x sign flips with it), small shift, gain/offset
    jitter (gradients scale with gain — they are linear in the image), pixel
    noise on intensity. Targets transform WITH the image or the label lies."""
    X, hm, wh, m = X.copy(), hm.copy(), wh.copy(), m.copy()
    B = len(X)
    flip = rng.random(B) < 0.5
    X[flip] = X[flip, :, :, ::-1]
    X[flip, 1] = -X[flip, 1]                       # grad_x under mirror
    hm[flip] = hm[flip, :, :, ::-1]
    wh[flip] = wh[flip, :, :, ::-1]
    m[flip] = m[flip, :, :, ::-1]
    sx = rng.integers(-4, 5, B); sy = rng.integers(-3, 4, B)
    for i in range(B):
        if sx[i] or sy[i]:
            X[i] = np.roll(X[i], (sy[i], sx[i]), axis=(1, 2))
            hs, hx = sy[i] // STRIDE, sx[i] // STRIDE
            for A in (hm, wh, m):
                A[i] = np.roll(A[i], (hs, hx), axis=(1, 2))
    g = rng.uniform(0.9, 1.1, (B, 1, 1, 1)).astype(np.float32)
    X *= g                                          # all channels linear in gain
    X[:, 0] += rng.uniform(-0.05, 0.05, (B, 1, 1)).astype(np.float32)
    X[:, 0] += rng.normal(0, 0.01, X[:, 0].shape).astype(np.float32)
    return X, hm, wh, m

class ThermalStudent(nn.Module):
    def __init__(self, n_ch=N_CH):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(n_ch, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.head_hm = nn.Conv2d(64, 1, 1)
        self.head_wh = nn.Conv2d(64, 2, 1)
    def forward(self, x):
        f = self.backbone(x)
        return self.head_hm(f), self.head_wh(f)

In [ ]:
def eval_thermal(model, X, hm, wh, mask, bs=64):
    """Hit-rate: predicted heatmap peak within 2 cells of a true center.
    Val is NEVER augmented - it answers 'how good on real frames'."""
    model.eval(); hits = tot = 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            hp, _ = model(torch.from_numpy(X[i:i+bs]).to(DEV))
            hp = torch.sigmoid(hp).cpu().numpy()
            for j in range(len(hp)):
                m = mask[i+j, 0]
                if m.sum() == 0: continue
                tot += 1
                py, px = np.unravel_index(hp[j, 0].argmax(), hp[j, 0].shape)
                ty, tx = np.where(m > 0)
                if np.min(np.hypot(ty-py, tx-px)) <= 2: hits += 1
    return hits / max(tot, 1)

def eval_thermal_metrics(model, X, mask, label_state, threshold=0.5, bs=64):
    """Frame detection metrics; UNKNOWN frames never enter a denominator."""
    model.eval(); scores = []; peaks = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            hp, _ = model(torch.from_numpy(np.ascontiguousarray(X[i:i+bs])).to(DEV))
            hp = torch.sigmoid(hp).cpu().numpy()[:, 0]
            scores.extend(hp.max(axis=(1, 2)).tolist())
            peaks.extend([np.unravel_index(h.argmax(), h.shape) for h in hp])
    tp = fp = fn = tn = neg_fp = 0
    for i, state in enumerate(label_state):
        if state < 0:
            continue
        pred = scores[i] >= threshold
        if state == 0:
            fp += int(pred); neg_fp += int(pred); tn += int(not pred); continue
        ty, tx = np.where(mask[i, 0] > 0)
        localized = bool(pred and len(tx) and
                         np.min(np.hypot(ty-peaks[i][0], tx-peaks[i][1])) <= 2)
        tp += int(localized); fn += int(not localized)
        fp += int(pred and not localized)
    precision = tp / max(tp + fp, 1); recall = tp / max(tp + fn, 1)
    return {'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/max(precision+recall, 1e-9),
            'false_positive_rate': neg_fp/max(neg_fp+tn, 1),
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

def train_thermal(channel_mask=None, augment=False, epochs=30, bs=64, tag='thermal'):
    tr, va = load_split('train'), load_split('val')
    Xt = build_channels(tr['scaled'], tr['dt_ms'])
    Xv = build_channels(va['scaled'], va['dt_ms'])
    if channel_mask is not None:
        mk = np.asarray(channel_mask, np.float32)[None, :, None, None]
        Xt, Xv = Xt*mk, Xv*mk
    nt = tr['n_th_boxes'].copy(); nt[tr['thermal_label_state'] != 1] = 0
    nv = va['n_th_boxes'].copy(); nv[va['thermal_label_state'] != 1] = 0
    hmt, wht, mt = make_targets(tr['th_boxes'], nt)
    hmv, whv, mv = make_targets(va['th_boxes'], nv)
    # Supervision hygiene (the run1 poison): a frame with a person whose label
    # did not reach grade-A must NOT train as 'no person'. Keep positives
    # (grade-A present) and independently VERIFIED empty frames;
    # drop the ambiguous rest. Applied AFTER build_channels so the temporal
    # diff is computed on the real sequence.
    keep = tr['thermal_label_state'] >= 0  # positive or VERIFIED negative
    Xt, hmt, wht, mt = Xt[keep], hmt[keep], wht[keep], mt[keep]
    print(f'[{tag}] training on {keep.sum()}/{len(keep)} frames '
          f'({int((tr["n_th_boxes"]>0).sum())} pos, ambiguous dropped)')
    model = ThermalStudent().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), 3e-4)
    rng = np.random.default_rng(0)
    order = np.arange(len(Xt))
    best, best_state = -1.0, None
    for ep in range(epochs):
        model.train(); rng.shuffle(order)
        for i in range(0, len(order), bs):
            k = order[i:i+bs]
            xb, hb, wb, mb = Xt[k], hmt[k], wht[k], mt[k]
            if augment:
                xb, hb, wb, mb = augment_thermal(xb, hb, wb, mb, rng)
            hp, wp = model(torch.from_numpy(np.ascontiguousarray(xb)).to(DEV))
            hb = torch.from_numpy(np.ascontiguousarray(hb)).to(DEV)
            wb = torch.from_numpy(np.ascontiguousarray(wb)).to(DEV)
            mb = torch.from_numpy(np.ascontiguousarray(mb)).to(DEV)
            # CenterNet penalty-reduced focal loss. Plain BCE on a 40x30 map
            # with one tiny gaussian collapsed to all-background on run1
            # (0.147 -> 0.088 while training); focal keeps the positives loud.
            p = torch.sigmoid(hp)
            pos = (hb > 0.999).float()
            hm_loss = -(pos*(1-p)**2*torch.log(p.clamp(1e-6))
                        + (1-pos)*(1-hb)**4*(p**2)*torch.log((1-p).clamp(1e-6))
                       ).sum() / pos.sum().clamp(1)
            loss = hm_loss + 0.1*(F.l1_loss(wp, wb, reduction='none')*mb).sum()/mb.sum().clamp(1)
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 5 == 0:
            metrics = eval_thermal_metrics(model, Xv, mv, va['thermal_label_state'])
            a = metrics['f1']
            if a > best:
                best = a
                best_state = {k2: v2.detach().cpu().clone()
                              for k2, v2 in model.state_dict().items()}
            print(f'[{tag}] ep {ep+1}: val F1 {a:.3f} P {metrics["precision"]:.3f} R {metrics["recall"]:.3f} (best {best:.3f})')
    if best_state is not None:
        model.load_state_dict(best_state)   # run1 also showed late-epoch decay
    print(f'[{tag}] FINAL best validation F1 {best:.3f}')
    return model, best

thermal_model, thermal_v1 = train_thermal(tag='thermal-V1')

In [ ]:
# Thermal ablation (V1 regime) - retrain from scratch per removed channel
thermal_results = {'full': thermal_v1}
for ci, name in enumerate(CH_NAMES):
    mk = [1.0]*N_CH; mk[ci] = 0.0
    _, acc = train_thermal(channel_mask=mk, tag=f'thermal-no-{name}')
    thermal_results[f'no-{name}'] = acc
print('\n=== thermal ablation (V1) ===')
for k, v in sorted(thermal_results.items(), key=lambda kv: -kv[1]):
    print(f'{k:16s} {v:.3f}  (drop {thermal_results["full"]-v:+.3f})')

## Radar Student V3 — multi-stream

Three independent streams, fused only after independent encoding:
**point cloud** (PointNet, max+mean pooling) · **Range-Doppler map** ·
**Range-Angle map** — each map expanded to 5 channels (original, two spatial
gradients, gradient magnitude, temporal gradient).

Streams activate by what the export actually carries: v1 = points only;
radar_people_ra.cfg sessions add RA; RD needs dedicated slow sessions (it is
off in the live cfg — 32 KB/frame does not fit the link). A missing map
disables its stream instead of stopping the run. Ablations cover point
features always, and map channels whenever the maps exist.

In [ ]:
# ============================================================
# RADAR STUDENT V3 — multi-stream: points + Range-Doppler + Range-Angle
# Streams are OPTIONAL by data availability:
#   export v1 sessions        -> points only
#   radar_people_ra.cfg data  -> points + RA (RD is off in that cfg: 32 KB/frame
#                                does not fit the live link; RD needs dedicated
#                                slow sessions)
# The model builds an encoder per available stream and fuses only what exists —
# the hard 'raise' of the draft became graceful degradation on purpose: the POC
# stop rule says a missing representation must not block training.
# ============================================================

MAX_OBJECTS = 4
RADAR_POINT_FEATURES = ['x', 'y', 'z', 'v', 'snr', 'noise']
MAP_CHANNELS = ['original', 'grad-axis0', 'grad-axis1', 'grad-mag', 'temporal']


def build_teacher_targets(split, max_objects=MAX_OBJECTS):
    nb = split['n_rgb_boxes']
    bx = split['rgb_boxes']
    state = split['radar_label_state']
    B = len(nb)
    targets = np.zeros((B, max_objects, 6), dtype=np.float32)
    for i in range(B):
        n = int(nb[i])
        if state[i] != 1:
            continue  # verified negative or unknown: no positive target boxes
        if n <= 0:
            continue
        objects = []
        for b in bx[i, :n]:
            x, y, w, h = b[:4]
            confidence = float(b[4]) if len(b) > 4 else 1.0
            objects.append([1.0, (x + 0.5*w)/RGB_W, (y + 0.5*h)/RGB_H,
                            w/RGB_W, h/RGB_H, confidence])
        objects.sort(key=lambda o: o[1])            # stable left -> right
        objects = objects[:max_objects]
        targets[i, :len(objects)] = np.asarray(objects, dtype=np.float32)
    return targets


def temporal_gradient(M):
    dt = np.zeros_like(M, dtype=np.float32)
    dt[1:] = M[1:] - M[:-1]
    return dt


def make_radar_map_channels(M, valid=None):
    """[B, H, W] map -> [B, 5, H, W]: original, d/axis0, d/axis1, |grad|, d/dt.

    valid (bool [B]) marks frames whose map really existed; a zero-filled
    invalid frame must not fake a temporal edge on its neighbours, so the
    temporal channel is zeroed wherever either side of the diff is invalid.
    """
    M = M.astype(np.float32)
    g0 = np.gradient(M, axis=1).astype(np.float32)
    g1 = np.gradient(M, axis=2).astype(np.float32)
    gmag = np.sqrt(g0**2 + g1**2).astype(np.float32)
    gt = temporal_gradient(M)
    if valid is not None:
        ok = valid.astype(bool)
        pair_ok = np.zeros_like(ok)
        pair_ok[1:] = ok[1:] & ok[:-1]
        gt[~pair_ok] = 0.0
    return np.stack([M, g0, g1, gmag, gt], axis=1).astype(np.float32)


def prepare_radar(split):
    points = split['radar'].copy().astype(np.float32)
    n_points = split['n_radar'].astype(np.int64)
    # v stays SIGNED. Honesty note: export v1 ran v_max 0.649 configs, a
    # walker folds (alias 1.298 m/s) and the sign is unreliable THERE; the
    # no-v ablation judges it, radar_people data makes the sign real.
    teacher = build_teacher_targets(split)
    rd = ra = None
    if 'range_doppler' in split:
        rd = make_radar_map_channels(split['range_doppler'].astype(np.float32),
                                     split.get('rd_valid'))
    if 'range_angle' in split:
        ra = make_radar_map_channels(split['range_angle'].astype(np.float32),
                                     split.get('ra_valid'))
    return points, n_points, rd, ra, teacher


def radar_point_statistics(R, n):
    valid = [R[i, :int(n[i])] for i in range(len(R)) if n[i] > 0]
    if not valid:
        return (np.zeros(R.shape[-1], np.float32),
                np.ones(R.shape[-1], np.float32))
    pts = np.concatenate(valid, axis=0)
    return (pts.mean(axis=0).astype(np.float32),
            np.maximum(pts.std(axis=0).astype(np.float32), 1e-4))


def normalize_radar_points(R, n, mean, std):
    R = R.copy()
    for i in range(len(R)):
        ni = int(n[i])
        if ni > 0:
            R[i, :ni] = (R[i, :ni] - mean) / std
    return R


def normalize_map_channels(M):
    if M is None:
        return None, None, None
    mean = M.mean(axis=(0, 2, 3), keepdims=True).astype(np.float32)
    std = np.maximum(M.std(axis=(0, 2, 3), keepdims=True).astype(np.float32), 1e-4)
    return ((M - mean) / std).astype(np.float32), mean, std


def apply_map_normalization(M, mean, std):
    if M is None:
        return None
    return ((M - mean) / std).astype(np.float32)


def augment_radar(points, n_points, teacher, rng):
    points, n_points, teacher = points.copy(), n_points.copy(), teacher.copy()
    B = len(points)
    # horizontal mirror: radar y -> -y, camera u -> 1-u, slots re-sorted
    # NOTE: the RA map is NOT mirrored here (angle axis would need flipping
    # too) - so mirror augmentation is applied to the point stream only when
    # maps are absent; train_radar disables it otherwise.
    flip = rng.random(B) < 0.5
    for i in np.where(flip)[0]:
        ni = int(n_points[i])
        if ni > 0:
            points[i, :ni, 1] *= -1.0
        valid = teacher[i, :, 0] > 0
        teacher[i, valid, 1] = 1.0 - teacher[i, valid, 1]
        objs = teacher[i, valid].copy()
        if len(objs) > 0:
            objs = objs[np.argsort(objs[:, 1])]
            teacher[i] = 0
            teacher[i, :len(objs)] = objs
    for i in range(B):
        ni = int(n_points[i])
        if ni <= 0:
            continue
        points[i, :ni, :3] += rng.normal(0.0, 0.03, (ni, 3)).astype(np.float32)
        points[i, :ni, 3] += rng.normal(0.0, 0.05, ni).astype(np.float32)
        points[i, :ni, 4] += rng.normal(0.0, 0.5, ni).astype(np.float32)
        points[i, :ni, 5] += rng.normal(0.0, 0.5, ni).astype(np.float32)
    for i in range(B):                        # REAL dropout: points removed
        ni = int(n_points[i])
        if ni <= 1:
            continue
        nk = max(1, int(round(ni * rng.uniform(0.70, 1.0))))
        selected = rng.choice(ni, size=nk, replace=False)
        kept = points[i, selected].copy()
        points[i] = 0
        points[i, :nk] = kept
        n_points[i] = nk
    return points, n_points, teacher


class RadarPointEncoder(nn.Module):
    def __init__(self, in_features=6, feat=128):
        super().__init__()
        self.point_encoder = nn.Sequential(
            nn.Linear(in_features, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Linear(128, feat), nn.LayerNorm(feat), nn.GELU())
        self.scene_encoder = nn.Sequential(
            nn.Linear(feat * 2, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, feat), nn.GELU())

    def forward(self, pts, n_pts):
        B, P, _ = pts.shape
        f = self.point_encoder(pts)
        idx = torch.arange(P, device=pts.device)[None, :]
        valid = idx < n_pts[:, None]
        f_max = f.masked_fill(~valid.unsqueeze(-1), -1e9).max(dim=1).values
        f_max = torch.where(n_pts[:, None] > 0, f_max, torch.zeros_like(f_max))
        masked = f * valid.unsqueeze(-1)
        f_mean = masked.sum(dim=1) / valid.sum(dim=1, keepdim=True).clamp(min=1)
        return self.scene_encoder(torch.cat([f_max, f_mean], dim=-1))


class RadarMapEncoder(nn.Module):
    def __init__(self, in_channels=5, feat=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Sequential(nn.Linear(128, feat), nn.LayerNorm(feat), nn.GELU())

    def forward(self, x):
        return self.fc(self.net(x).flatten(1))


class RadarStudent(nn.Module):
    """Streams fused only AFTER independent encoding. use_rd/use_ra reflect
    what the dataset actually carries."""

    def __init__(self, point_feat=128, map_feat=128, fusion_feat=256,
                 max_objects=MAX_OBJECTS, use_rd=False, use_ra=False):
        super().__init__()
        self.max_objects = max_objects
        self.use_rd, self.use_ra = use_rd, use_ra
        self.point_encoder = RadarPointEncoder(6, point_feat)
        total = point_feat
        if use_rd:
            self.rd_encoder = RadarMapEncoder(5, map_feat)
            total += map_feat
        if use_ra:
            self.ra_encoder = RadarMapEncoder(5, map_feat)
            total += map_feat
        self.fusion = nn.Sequential(
            nn.Linear(total, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, fusion_feat), nn.LayerNorm(fusion_feat), nn.GELU())
        self.object_head = nn.Sequential(
            nn.Linear(fusion_feat, 256), nn.GELU(),
            nn.Linear(256, max_objects * 5))

    def forward(self, points, n_points, range_doppler=None, range_angle=None):
        latents = [self.point_encoder(points, n_points)]
        rd_latent = ra_latent = None
        if self.use_rd:
            rd_latent = self.rd_encoder(range_doppler)
            latents.append(rd_latent)
        if self.use_ra:
            ra_latent = self.ra_encoder(range_angle)
            latents.append(ra_latent)
        radar_latent = self.fusion(torch.cat(latents, dim=1))
        out = self.object_head(radar_latent).view(-1, self.max_objects, 5)
        return {'radar_latent': radar_latent, 'point_latent': latents[0],
                'rd_latent': rd_latent, 'ra_latent': ra_latent,
                'object_logits': out[..., 0],
                'boxes': torch.sigmoid(out[..., 1:5])}


def radar_student_loss(output, teacher):
    presence = teacher[..., 0]
    teacher_boxes = teacher[..., 1:5]
    confidence = teacher[..., 5]
    presence_loss = F.binary_cross_entropy_with_logits(
        output['object_logits'], presence, reduction='none')
    weight = torch.where(presence > 0, confidence.clamp(min=0.25),
                         torch.ones_like(confidence))
    presence_loss = (presence_loss * weight).mean()
    mask = presence.unsqueeze(-1)
    box_error = F.smooth_l1_loss(output['boxes'], teacher_boxes, reduction='none')
    box_weight = mask * confidence.unsqueeze(-1)
    box_loss = (box_error * box_weight).sum() / box_weight.sum().clamp(min=1)
    center_error = torch.abs(output['boxes'][..., :2] - teacher_boxes[..., :2])
    center_weight = box_weight[..., :2]
    center_loss = (center_error * center_weight).sum() \
        / center_weight.sum().clamp(min=1)
    total = presence_loss + 2.0 * center_loss + 0.5 * box_loss
    return {'total': total, 'presence': presence_loss,
            'center': center_loss, 'box': box_loss}


def _to_dev(a):
    return torch.from_numpy(np.ascontiguousarray(a)).to(DEV)


def evaluate_radar(model, points, n_points, rd, ra, teacher):
    model.eval()
    with torch.no_grad():
        output = model(_to_dev(points), _to_dev(n_points),
                       _to_dev(rd) if rd is not None else None,
                       _to_dev(ra) if ra is not None else None)
    logits = output['object_logits'].cpu().numpy()
    boxes = output['boxes'].cpu().numpy()
    pred_presence = logits > 0
    gt_presence = teacher[..., 0] > 0.5
    presence_acc = float((pred_presence == gt_presence).mean())
    pred_frame = pred_presence.any(axis=1); gt_frame = gt_presence.any(axis=1)
    tp = int((pred_frame & gt_frame).sum()); fp = int((pred_frame & ~gt_frame).sum())
    fn = int((~pred_frame & gt_frame).sum()); tn = int((~pred_frame & ~gt_frame).sum())
    precision = tp/max(tp+fp, 1); recall = tp/max(tp+fn, 1)
    f1 = 2*precision*recall/max(precision+recall, 1e-9)
    positive = gt_presence
    if positive.any():
        du = np.abs(boxes[..., 0] - teacher[..., 1])[positive] * RGB_W
        dv = np.abs(boxes[..., 1] - teacher[..., 2])[positive] * RGB_H
        median_u_px, median_v_px = float(np.median(du)), float(np.median(dv))
    else:
        median_u_px = median_v_px = float('nan')
    return {'presence_acc': presence_acc, 'precision': precision,
            'recall': recall, 'f1': f1, 'tp': tp, 'fp': fp,
            'fn': fn, 'tn': tn,
            'median_u_px': median_u_px, 'median_v_px': median_v_px}


def train_radar(epochs=60, bs=128, augment=True, point_feature_mask=None,
                rd_channel_mask=None, ra_channel_mask=None,
                tag='radar-multistream'):
    tr, va = load_split('train'), load_split('val')
    Rt, nt, RDt, RAt, Tt = prepare_radar(tr)
    Rv, nv, RDv, RAv, Tv = prepare_radar(va)
    kt = tr['radar_label_state'] >= 0
    kv = va['radar_label_state'] >= 0
    Rt, nt, Tt = Rt[kt], nt[kt], Tt[kt]
    Rv, nv, Tv = Rv[kv], nv[kv], Tv[kv]
    RDt = RDt[kt] if RDt is not None else None
    RAt = RAt[kt] if RAt is not None else None
    RDv = RDv[kv] if RDv is not None else None
    RAv = RAv[kv] if RAv is not None else None
    if not len(Rt) or not len(Rv):
        raise ValueError('radar training needs known positives and/or --verified-negative sessions')
    use_rd = RDt is not None and RDv is not None
    use_ra = RAt is not None and RAv is not None
    print(f'[{tag}] streams: points'
          + (' + range-doppler' if use_rd else '')
          + (' + range-angle' if use_ra else '')
          + ('' if (use_rd or use_ra) else '   (maps absent in this export)'))
    # mirror augmentation flips the point cloud but not the maps - with maps
    # active the two streams would describe different scenes, so it is off
    do_augment = augment and not (use_rd or use_ra)
    if augment and not do_augment:
        print(f'[{tag}] mirror/noise augmentation disabled (maps present)')

    point_mean, point_std = radar_point_statistics(Rt, nt)
    Rt = normalize_radar_points(Rt, nt, point_mean, point_std)
    Rv = normalize_radar_points(Rv, nv, point_mean, point_std)
    RDt, rd_mean, rd_std = normalize_map_channels(RDt)
    RAt, ra_mean, ra_std = normalize_map_channels(RAt)
    RDv = apply_map_normalization(RDv, rd_mean, rd_std)
    RAv = apply_map_normalization(RAv, ra_mean, ra_std)

    if point_feature_mask is not None:
        mk = np.asarray(point_feature_mask, np.float32)[None, None, :]
        Rt *= mk
        Rv *= mk
    if rd_channel_mask is not None and use_rd:
        mk = np.asarray(rd_channel_mask, np.float32)[None, :, None, None]
        RDt *= mk
        RDv *= mk
    if ra_channel_mask is not None and use_ra:
        mk = np.asarray(ra_channel_mask, np.float32)[None, :, None, None]
        RAt *= mk
        RAv *= mk

    model = RadarStudent(use_rd=use_rd, use_ra=use_ra).to(DEV)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    rng = np.random.default_rng(0)
    order = np.arange(len(Rt))

    for ep in range(epochs):
        model.train()
        rng.shuffle(order)
        epoch_loss = 0.0
        for i in range(0, len(order), bs):
            k = order[i:i + bs]
            pb, nb2, tb = Rt[k].copy(), nt[k].copy(), Tt[k].copy()
            if do_augment:
                pb, nb2, tb = augment_radar(pb, nb2, tb, rng)
            output = model(_to_dev(pb), _to_dev(nb2),
                           _to_dev(RDt[k]) if use_rd else None,
                           _to_dev(RAt[k]) if use_ra else None)
            losses = radar_student_loss(output, _to_dev(tb))
            optimizer.zero_grad()
            losses['total'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += float(losses['total'].item())
        scheduler.step()
        if ep == 0 or (ep + 1) % 10 == 0:
            print(f'[{tag}] epoch={ep+1:03d} loss={epoch_loss:.4f}')

    result = evaluate_radar(model, Rv, nv, RDv, RAv, Tv)
    print(f'\n[{tag}] presence={result["presence_acc"]:.3f} '
          f'u-error={result["median_u_px"]:.1f}px '
          f'v-error={result["median_v_px"]:.1f}px')
    model.point_mean, model.point_std = point_mean, point_std
    model.rd_mean, model.rd_std = rd_mean, rd_std
    model.ra_mean, model.ra_std = ra_mean, ra_std
    model.streams = {'rd': use_rd, 'ra': use_ra}
    return model, result


radar_model, radar_full_result = train_radar(tag='radar-full')
full_acc = radar_full_result['f1']

# ---- point feature ablation ------------------------------------------------
point_ablation_results = {'full': radar_full_result}
for ci, name in enumerate(RADAR_POINT_FEATURES):
    mask = [1.0] * 6
    mask[ci] = 0.0
    _, result = train_radar(point_feature_mask=mask, tag=f'radar-no-point-{name}')
    point_ablation_results[f'no-{name}'] = result
print('\n=== POINT FEATURE ABLATION ===')
for name, result in point_ablation_results.items():
    print(f'{name:15s} presence={result["presence_acc"]:.3f} '
          f'F1={result["f1"]:.3f} P={result["precision"]:.3f} R={result["recall"]:.3f} '
          f'drop={full_acc - result["f1"]:+.3f} '
          f'u={result["median_u_px"]:.1f}px v={result["median_v_px"]:.1f}px')

# ---- map channel ablations run only when the export carries the maps -------
_probe = load_split('val')
if 'range_doppler' in _probe:
    rd_ablation_results = {}
    for ci, name in enumerate(MAP_CHANNELS):
        mask = [1.0] * 5
        mask[ci] = 0.0
        _, result = train_radar(rd_channel_mask=mask, tag=f'no-rd-{name}')
        rd_ablation_results[f'no-rd-{name}'] = result
    print('\n=== RANGE-DOPPLER ABLATION ===')
    for name, result in rd_ablation_results.items():
        print(f'{name:22s} presence={result["presence_acc"]:.3f} '
              f'F1={result["f1"]:.3f} drop={full_acc - result["f1"]:+.3f}')
if 'range_angle' in _probe:
    ra_ablation_results = {}
    for ci, name in enumerate(MAP_CHANNELS):
        mask = [1.0] * 5
        mask[ci] = 0.0
        _, result = train_radar(ra_channel_mask=mask, tag=f'no-ra-{name}')
        ra_ablation_results[f'no-ra-{name}'] = result
    print('\n=== RANGE-ANGLE ABLATION ===')
    for name, result in ra_ablation_results.items():
        print(f'{name:22s} presence={result["presence_acc"]:.3f} '
              f'F1={result["f1"]:.3f} drop={full_acc - result["f1"]:+.3f}')
del _probe


## Training V2 (thermal) + summary

Thermal V2 = augmented training on the same data (mirror with grad-x sign fix,
shifts, gain/noise jitter). Val is never augmented. The radar section above is
already the V2 architecture with augmentation built in.

In [ ]:
# ---- summary + save everything the Jetson needs ---------------------------
thermal_model_v2, thermal_v2 = train_thermal(augment=True, epochs=45, tag='thermal-V2')

print('\n=== SUMMARY ===')
print(f'thermal F1       : V1 {thermal_v1:.3f} -> V2 {thermal_v2:.3f} ({thermal_v2-thermal_v1:+.3f})')
print(f'radar multistream: presence {radar_full_result["presence_acc"]:.3f}, '
      f'u-err {radar_full_result["median_u_px"]:.1f}px, '
      f'v-err {radar_full_result["median_v_px"]:.1f}px')

best_thermal = thermal_model_v2 if thermal_v2 >= thermal_v1 else thermal_model
torch.save(best_thermal.state_dict(), os.path.join(DATA, 'thermal_student.pt'))
torch.save({'state_dict': radar_model.state_dict(),
            'point_mean': radar_model.point_mean, 'point_std': radar_model.point_std,
            'rd_mean': radar_model.rd_mean, 'rd_std': radar_model.rd_std,
            'ra_mean': radar_model.ra_mean, 'ra_std': radar_model.ra_std,
            'streams': radar_model.streams, 'max_objects': MAX_OBJECTS},
           os.path.join(DATA, 'radar_student.pt'))
print('saved to Drive: thermal_student.pt, radar_student.pt')

## Reading the results

- A channel whose removal costs nothing was either learned internally (fine
  for derived channels) or unexercised by v1 — collect the hard sessions
  before believing a null result.
- v1 is indoor/short-range: expect Doppler to matter little on holds and much
  more on the walking campaign data.
- The V1→V2 delta bounds what augmentation buys; the data-v2 campaign
  (7–15 m, RA heatmaps, calibrated °C) is the bigger lever and gets its own
  export version so the comparison stays clean.